# Regresión Lineal Múltiple
### Predicción del costo del seguro médico

**Dataset:** insurance.csv — 1,338 personas  
**Columnas disponibles:** age, sex, bmi, children, smoker, region, charges  
**Variable objetivo (Y):** charges (costo del seguro)

**Flujo del notebook:**
1. Cargar datos
2. Codificar variables categóricas
3. Analizar correlación de Pearson
4. Normalizar
5. Entrenar modelo
6. Evaluar métricas
7. Predecir nuevo cliente

In [1]:
# ============================================================
# 1. IMPORTAR LIBRERÍAS
# ============================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# ============================================================
# 2. CARGAR DATOS
# ============================================================
df = pd.read_csv("insurance.csv")

print("Primeras filas:")
print(df.head())
print("\nDimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

Primeras filas:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

Dimensiones: (1338, 7)

Tipos de datos:
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object


In [3]:
# ============================================================
# 3. CODIFICAR VARIABLES CATEGÓRICAS
# ============================================================
# Los modelos de ML solo entienden números, no texto.
# Hay dos formas de codificar según cuántas categorías tenga la columna:
#
#   Binarización .map()  → para columnas con solo 2 valores
#       sex:    'male' → 1  |  'female' → 0
#       smoker: 'yes'  → 1  |  'no'     → 0
#
#   One-Hot Encoding get_dummies() → para columnas con 3+ valores
#       region: tiene 4 categorías (northeast, northwest, southeast, southwest)
#       Crea una columna por categoría, con 1 si aplica y 0 si no.
#       Así el modelo no asume que hay un orden entre las regiones.

df_enc = df.copy()

df_enc['sex']    = df_enc['sex'].map({'male': 1, 'female': 0})
df_enc['smoker'] = df_enc['smoker'].map({'yes': 1, 'no': 0})
df_enc = pd.get_dummies(df_enc, columns=['region'], dtype=int)

print("Columnas después de codificar:")
print(df_enc.columns.tolist())
print("\nPrimeras filas:")
print(df_enc.head())

Columnas después de codificar:
['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']

Primeras filas:
   age  sex     bmi  children  smoker      charges  region_northeast  \
0   19    0  27.900         0       1  16884.92400                 0   
1   18    1  33.770         1       0   1725.55230                 0   
2   28    1  33.000         3       0   4449.46200                 0   
3   33    1  22.705         0       0  21984.47061                 0   
4   32    1  28.880         0       0   3866.85520                 0   

   region_northwest  region_southeast  region_southwest  
0                 0                 0                 1  
1                 0                 1                 0  
2                 0                 1                 0  
3                 1                 0                 0  
4                 1                 0                 0  


In [4]:
# ============================================================
# 4. CORRELACIÓN DE PEARSON
# ============================================================
# Calculamos qué tan relacionada está cada variable con 'charges'.
# Esto nos dice cuáles variables tienen más peso en la predicción.
#
# Escala de r (valor absoluto):
#   0.8 - 1.0 → muy alta  |  0.6 - 0.8 → alta
#   0.4 - 0.6 → moderada  |  0.2 - 0.4 → baja
#   0.0 - 0.2 → muy baja  |  0.0       → nula

correlacion = df_enc.corr()['charges'].drop('charges').sort_values(ascending=False)

print("Correlación de cada variable con charges:")
print("=" * 45)
for col, val in correlacion.items():
    barra = '█' * int(abs(val) * 30)
    signo = '+' if val >= 0 else '-'
    print(f"{col:<22} {signo}{abs(val):.3f}  {barra}")

print("\n→ smoker tiene correlación muy alta (0.787): es el factor más importante")
print("→ age y bmi tienen correlación baja-moderada")
print("→ region, children y sex tienen correlación casi nula")

Correlación de cada variable con charges:
smoker                 +0.787  ███████████████████████
age                    +0.299  ████████
bmi                    +0.198  █████
region_southeast       +0.074  ██
children               +0.068  ██
sex                    +0.057  █
region_northeast       +0.006  
region_northwest       -0.040  █
region_southwest       -0.043  █

→ smoker tiene correlación muy alta (0.787): es el factor más importante
→ age y bmi tienen correlación baja-moderada
→ region, children y sex tienen correlación casi nula


In [5]:
# ============================================================
# 5. SELECCIONAR VARIABLES PARA EL MODELO
# ============================================================
# Usamos las 4 variables pedidas por la tarea.
# La correlación confirmó que smoker, age y bmi son las más útiles.
# sex se incluye porque la tarea lo pide (su correlación es baja pero no cero).

features = ['age', 'bmi', 'smoker', 'sex']

X = df_enc[features].values
y = df_enc['charges'].values

print("Variables de entrada (X):", features)
print("Variable a predecir  (y): charges")
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Variables de entrada (X): ['age', 'bmi', 'smoker', 'sex']
Variable a predecir  (y): charges
Forma de X: (1338, 4)
Forma de y: (1338,)


In [6]:
# ============================================================
# 6. NORMALIZACIÓN MIN-MAX
# ============================================================
# Escala todo al rango [0, 1].
# Fórmula: (valor - mínimo) / (máximo - mínimo)
#
# Solo normalizamos age (col 0) y bmi (col 1).
# smoker y sex ya son 0/1, no necesitan normalizarse.
# También normalizamos y (charges) para que el modelo trabaje mejor.
#
# IMPORTANTE: guardamos los min/max para desnormalizar después.

edad_min, edad_max = X[:, 0].min(), X[:, 0].max()
bmi_min,  bmi_max  = X[:, 1].min(), X[:, 1].max()
y_min,    y_max    = y.min(),        y.max()

X[:, 0] = (X[:, 0] - edad_min) / (edad_max - edad_min)
X[:, 1] = (X[:, 1] - bmi_min)  / (bmi_max  - bmi_min)
y       = (y - y_min) / (y_max - y_min)

print("Normalización aplicada.")
print("Rango age (norm): ", round(X[:, 0].min(), 2), "-", round(X[:, 0].max(), 2))
print("Rango bmi (norm): ", round(X[:, 1].min(), 2), "-", round(X[:, 1].max(), 2))
print("Rango y   (norm): ", round(y.min(),       2), "-", round(y.max(),       2))

Normalización aplicada.
Rango age (norm):  0.0 - 1.0
Rango bmi (norm):  0.0 - 1.0
Rango y   (norm):  0.0 - 1.0


In [7]:
# ============================================================
# 7. DIVIDIR EN ENTRENAMIENTO Y PRUEBA (80% / 20%)
# ============================================================
# 80% → el modelo aprende con estos datos
# 20% → evaluamos qué tan bien predice con datos que nunca vio
# random_state=42 → fija el "azar" para resultados reproducibles

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Entrenamiento:", X_train.shape[0], "muestras")
print("Prueba:        ", X_test.shape[0],  "muestras")

Entrenamiento: 1070 muestras
Prueba:         268 muestras


In [8]:
# ============================================================
# 8. CREAR Y ENTRENAR EL MODELO
# ============================================================
# LinearRegression encuentra los coeficientes (w) óptimos
# usando Mínimos Cuadrados: minimiza el error al cuadrado.

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Intercepto (b):", round(modelo.intercept_, 4))
print("\nCoeficientes (w) — mayor valor = más influencia:")
for nombre, coef in zip(features, modelo.coef_):
    print(f"  {nombre:<10}: {round(coef, 4)}")

Intercepto (b): -0.0472

Coeficientes (w) — mayor valor = más influencia:
  age       : 0.1905
  bmi       : 0.1937
  smoker    : 0.3779
  sex       : 0.0001


In [9]:
# ============================================================
# 9. EVALUAR CON MÉTRICAS
# ============================================================
# MAE (Mean Absolute Error)  → error promedio, fácil de interpretar
# MSE (Mean Squared Error)   → penaliza más los errores grandes
# R²  (Coef. Determinación)  → % de variación explicada por el modelo
#                               Cercano a 1 = buen modelo

predicciones = modelo.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
mse = mean_squared_error(y_test, predicciones)
r2  = r2_score(y_test, predicciones)

print("============================")
print("MÉTRICAS DEL MODELO")
print("============================")
print(f"MAE:  {round(mae, 4)}")
print(f"MSE:  {round(mse, 4)}")
print(f"R²:   {round(r2, 4)}")
print(f"\n→ El modelo explica el {round(r2 * 100, 1)}% de la variación en el costo.")

MÉTRICAS DEL MODELO
MAE:  0.068
MSE:  0.0088
R²:   0.7777

→ El modelo explica el 77.8% de la variación en el costo.


In [10]:
# ============================================================
# 10. PREDECIR UN NUEVO CLIENTE
# ============================================================
edad_nueva    = 30
bmi_nuevo     = 28
fumador_nuevo = 1    # 1 = sí fuma  | 0 = no fuma
genero_nuevo  = 1    # 1 = masculino | 0 = femenino

# Normalizar con los mismos min/max del dataset original
edad_norm = (edad_nueva - edad_min) / (edad_max - edad_min)
bmi_norm  = (bmi_nuevo  - bmi_min)  / (bmi_max  - bmi_min)

nuevo_cliente = [[edad_norm, bmi_norm, fumador_nuevo, genero_nuevo]]

# Predecir (resultado normalizado) y luego desnormalizar
pred_norm = modelo.predict(nuevo_cliente)
pred_real = (pred_norm[0] * (y_max - y_min)) + y_min

print("============================")
print("NUEVO CLIENTE")
print("============================")
print(f"Edad:    {edad_nueva}")
print(f"BMI:     {bmi_nuevo}")
print(f"Fumador: {'Sí' if fumador_nuevo == 1 else 'No'}")
print(f"Género:  {'Masculino' if genero_nuevo == 1 else 'Femenino'}")
print(f"\nCosto estimado del seguro: ${round(pred_real, 2)}")

NUEVO CLIENTE
Edad:    30
BMI:     28
Fumador: Sí
Género:  Masculino

Costo estimado del seguro: $28894.41
